In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Load the dataset
df = pd.read_csv('electricity_bill_dataset.csv')


In [ ]:
# Exploratory Data Analysis (EDA)
print('First 5 rows:')
print(df.head())
print('\n' + '='*50)
print('Dataset Info:')
print(df.info())
print('\n' + '='*50)
print('Statistical Summary:')
print(df.describe())
print('\n' + '='*50)
print('Missing Values:')
print(df.isnull().sum())


In [ ]:
# Calculate correlation matrix for numeric features
numeric_df = df.select_dtypes(include=['number'])
corr = numeric_df.corr()

# Plot correlation matrix
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.3f', cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix - Electricity Bill Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlation Matrix:')
print(corr)


In [ ]:
# Identify the feature with highest correlation to target variable (ElectricityBill)
target = 'ElectricityBill'
corr_target = corr[target].sort_values(ascending=False)

print('Correlation with Target Variable (ElectricityBill):')
print(corr_target)

# Select the feature with highest correlation (excluding the target itself)
best_feature = corr_target.index[1]  # Index 0 is the target itself
highest_corr = corr_target.iloc[1]

print(f'\nSelected Feature: {best_feature}')
print(f'Correlation with Target: {highest_corr:.4f}')


In [ ]:
# Manual Simple Linear Regression Implementation
# Extract feature and target
X = df[best_feature].values
Y = df[target].values

# Calculate means
x_mean = np.mean(X)
y_mean = np.mean(Y)

# Manual calculation of slope (b1) and intercept (b0)
# Formula: b1 = Σ((X - X_mean)(Y - Y_mean)) / Σ((X - X_mean)²)
#          b0 = Y_mean - b1 * X_mean

numerator = np.sum((X - x_mean) * (Y - y_mean))
denominator = np.sum((X - x_mean)**2)
b1 = numerator / denominator
b0 = y_mean - b1 * x_mean

print('='*60)
print('MANUAL LINEAR REGRESSION CALCULATIONS')
print('='*60)
print(f'Feature: {best_feature}')
print(f'Target: {target}')
print(f'\nNumber of samples: {len(X)}')
print(f'X mean: {x_mean:.4f}')
print(f'Y mean: {y_mean:.4f}')
print(f'\nManually Calculated Slope (b1): {b1:.6f}')
print(f'Manually Calculated Intercept (b0): {b0:.6f}')
print(f'\nLinear Equation: Y = {b0:.4f} + {b1:.6f} * X')


In [ ]:
# Make predictions using manually calculated coefficients
Y_pred_manual = b0 + b1 * X

print('\nManual Predictions (first 10 samples):')
for i in range(10):
    print(f'X={X[i]:.2f}, Actual Y={Y[i]:.2f}, Predicted Y={Y_pred_manual[i]:.2f}, Error={Y[i]-Y_pred_manual[i]:.2f}')


In [ ]:
# Calculate Mean Squared Error (MSE) and Cost Function for manual model
# MSE = (1/n) * Σ(Y - Y_pred)²

mse_manual = np.mean((Y - Y_pred_manual)**2)
rmse_manual = np.sqrt(mse_manual)
cost_manual = mse_manual  # Cost function is the same as MSE

print('\n' + '='*60)
print('MANUAL MODEL PERFORMANCE METRICS')
print('='*60)
print(f'Mean Squared Error (MSE): {mse_manual:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse_manual:.4f}')
print(f'Cost Function (MSE): {cost_manual:.4f}')


In [ ]:
# Linear Regression using scikit-learn
# Reshape X for sklearn (requires 2D array)
X_reshaped = X.reshape(-1, 1)

# Create and train the model
model_sklearn = LinearRegression()
model_sklearn.fit(X_reshaped, Y)

# Extract coefficients
b1_sklearn = model_sklearn.coef_[0]
b0_sklearn = model_sklearn.intercept_

print('\n' + '='*60)
print('SCIKIT-LEARN LINEAR REGRESSION')
print('='*60)
print(f'Slope (b1) from sklearn: {b1_sklearn:.6f}')
print(f'Intercept (b0) from sklearn: {b0_sklearn:.6f}')
print(f'Linear Equation: Y = {b0_sklearn:.4f} + {b1_sklearn:.6f} * X')


In [ ]:
# Make predictions using sklearn model
Y_pred_sklearn = model_sklearn.predict(X_reshaped)

# Calculate MSE for sklearn model
mse_sklearn = mean_squared_error(Y, Y_pred_sklearn)
rmse_sklearn = np.sqrt(mse_sklearn)
cost_sklearn = mse_sklearn

print('\n' + '='*60)
print('SCIKIT-LEARN MODEL PERFORMANCE METRICS')
print('='*60)
print(f'Mean Squared Error (MSE): {mse_sklearn:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse_sklearn:.4f}')
print(f'Cost Function (MSE): {cost_sklearn:.4f}')


In [ ]:
# Compare manually calculated values with scikit-learn implementation
print('\n' + '='*60)
print('COMPARISON: MANUAL vs SCIKIT-LEARN')
print('='*60)

print(f'\nCoefficient Comparison:')
print(f'  Slope (b1):     Manual = {b1:.6f}, sklearn = {b1_sklearn:.6f}, Difference = {abs(b1 - b1_sklearn):.10f}')
print(f'  Intercept (b0): Manual = {b0:.6f}, sklearn = {b0_sklearn:.6f}, Difference = {abs(b0 - b0_sklearn):.10f}')

print(f'\nPerformance Metrics Comparison:')
print(f'  MSE:  Manual = {mse_manual:.4f}, sklearn = {mse_sklearn:.4f}, Difference = {abs(mse_manual - mse_sklearn):.6f}')
print(f'  RMSE: Manual = {rmse_manual:.4f}, sklearn = {rmse_sklearn:.4f}, Difference = {abs(rmse_manual - rmse_sklearn):.6f}')

# MSE between manual and sklearn predictions
mse_between = mean_squared_error(Y_pred_manual, Y_pred_sklearn)
print(f'\nMSE between Manual and sklearn predictions: {mse_between:.10f}')

if mse_between < 1e-8:
    print('\nResult: Both implementations match perfectly!')
else:
    print('\nResult: Small differences detected (likely due to rounding)')


In [ ]:
# Visualize the regression line
plt.figure(figsize=(14, 5))

# Plot 1: Manual Regression
plt.subplot(1, 2, 1)
plt.scatter(X, Y, alpha=0.3, s=10, label='Actual Data')
plt.plot(X, Y_pred_manual, color='red', linewidth=2, label='Manual Linear Regression')
plt.xlabel(best_feature, fontsize=11)
plt.ylabel(target, fontsize=11)
plt.title('Manual Linear Regression', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Sklearn Regression
plt.subplot(1, 2, 2)
plt.scatter(X, Y, alpha=0.3, s=10, label='Actual Data')
plt.plot(X, Y_pred_sklearn, color='green', linewidth=2, label='Scikit-learn Linear Regression')
plt.xlabel(best_feature, fontsize=11)
plt.ylabel(target, fontsize=11)
plt.title('Scikit-learn Linear Regression', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Residuals Analysis
residuals_manual = Y - Y_pred_manual
residuals_sklearn = Y - Y_pred_sklearn

plt.figure(figsize=(14, 5))

# Residuals plot for manual model
plt.subplot(1, 2, 1)
plt.scatter(Y_pred_manual, residuals_manual, alpha=0.3, s=10)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Values', fontsize=11)
plt.ylabel('Residuals', fontsize=11)
plt.title('Residuals Plot - Manual Model', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)

# Residuals plot for sklearn model
plt.subplot(1, 2, 2)
plt.scatter(Y_pred_sklearn, residuals_sklearn, alpha=0.3, s=10)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Values', fontsize=11)
plt.ylabel('Residuals', fontsize=11)
plt.title('Residuals Plot - Scikit-learn Model', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Residuals Statistics (Manual Model):')
print(f'  Mean: {np.mean(residuals_manual):.6f}')
print(f'  Std Dev: {np.std(residuals_manual):.4f}')


In [ ]:
# Final Summary
print('\n' + '='*70)
print('FINAL SUMMARY - SIMPLE LINEAR REGRESSION ANALYSIS')
print('='*70)
print(f'\nDataset: Electricity Bill Dataset')
print(f'Target Variable: {target}')
print(f'Feature Selected: {best_feature} (Correlation: {highest_corr:.4f})')
print(f'Number of Samples: {len(X)}')
print(f'\nREGRESSION EQUATIONS:')
print(f'  Manual: Y = {b0:.4f} + {b1:.6f} * {best_feature}')
print(f'  Sklearn: Y = {b0_sklearn:.4f} + {b1_sklearn:.6f} * {best_feature}')
print(f'\nMODEL PERFORMANCE:')
print(f'  Manual Model  - MSE: {mse_manual:.4f}, RMSE: {rmse_manual:.4f}')
print(f'  Sklearn Model - MSE: {mse_sklearn:.4f}, RMSE: {rmse_sklearn:.4f}')
print(f'\nBoth implementations produce identical results!')
print('='*70)
